In [1]:
import librosa
import librosa.display
import numpy as np
import pandas as pd
import os
from typing import List, Tuple, Dict
import matplotlib.pyplot as plt
import random
import cv2
# from vgg16 import VGG16_LSTM
import pickle
import torch.nn as nn
import torch
import datetime
import torch.nn.functional as F
import torchvision.models as models
from torchvision import transforms, utils, models
from torch.utils.data import Dataset, DataLoader
from facenet_pytorch import MTCNN
import logging
import torch
import torch.nn as nn
from vivit_probere1 import ViViTBackbone, ViViTLinearProbe
import gc
from einops import rearrange
from tqdm import tqdm
# from kw_r2plus1d_video_swin_transformer import SwinTransformer3D
batchsz = 4
frame_nums=15
start_epoch = 0
test_flage=False
model_name ="fullshot"
logger = logging.getLogger(__name__)
logger.setLevel(logging.DEBUG)
formatter = logging.Formatter('[%(asctime)s]::%(module)s::%(levelname)s::%(message)s')
streamHandler = logging.StreamHandler()
streamHandler.setFormatter(formatter)
fileHandler = logging.FileHandler('./LOG/personalityLog.log')
fileHandler.setFormatter(formatter)
logger.addHandler(streamHandler)
logger.addHandler(fileHandler)
USE_CUDA = torch.cuda.is_available()
# device = torch.device('cuda:1' if USE_CUDA else 'cpu')

In [2]:
logger.debug('============={}+Video Vision Transformer Strat!============='.format(model_name))
USE_CUDA = torch.cuda.is_available()
print(USE_CUDA)

device = torch.device('cuda:0' if USE_CUDA else 'cpu')
print('학습을 진행하는 기기:', device)

[2026-09-08 15:34:16,511]::3184030073::DEBUG::=============fullshot+Video Vision Transformer Strat!=============


True
학습을 진행하는 기기: cuda:0


In [3]:
def getdata(pathname,datatype):
    if datatype=="train":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            # train_data_set.extend(pickle.load(output_file))
            final_data_set.extend(pickle.load(output_file))
            # print(i)
        return final_data_set
    elif datatype=="valid":
        final_data_set=[]
        output_file=open(pathname, "rb")
        for i in range(1):
            valid_data_set=[]
            final_data_set.extend(pickle.load(output_file))
            del valid_data_set
        return final_data_set
print(1)
def reshape_to_expected_input(dataset: List[Tuple[ np.ndarray, np.ndarray]]) -> Tuple[
     np.ndarray, np.ndarray]:
    x0_list = []
    x1_list = []
    for i in range(0, len(dataset)):
        x0_list.append(dataset[i][0])
        x1_list.append(dataset[i][1])
    return (np.stack(x0_list), np.stack(x1_list))

1


In [4]:

save_model_file_path = './save_swintransformer_folder/{}_{}.{}'
mtcnn= MTCNN(image_size=224, margin=0, min_face_size= 20, thresholds=[0.6, 0.7, 0.7], post_process=True)
print(1)

1


In [5]:
validation_set_data=getdata("/data/qx/datafrom136/data/15Frames/Full_frames/valid_set.dat","valid")
validation_set_data=reshape_to_expected_input(validation_set_data)
print(1)

1


In [6]:
train_set_data=getdata("/data/qx/datafrom136/data/15Frames/Full_frames/train_set.dat","train")
train_set_data=reshape_to_expected_input(train_set_data)
print(1)

1


In [7]:
transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor(),
])
print(1)

1


In [8]:
class ChalearnDataset(Dataset):
    def __init__(self,fullshot,tagdata,transform=None):
        self.fullshot=fullshot
        self.tagdata=tagdata
        self.transform = transform  # 표준화 여부
    def __len__(self):
        return len(self.fullshot)

    def __getitem__(self, idx):
        if torch.is_tensor(idx):
            idx = idx.tolist()#텐서의 경우 목록으로 돌아가기
        fullshot=self.fullshot[idx]
        fullshot=torch.FloatTensor(fullshot)
        big_five_sorces=self.tagdata[idx]
        big_five_sorces = torch.FloatTensor(big_five_sorces)
        return fullshot,big_five_sorces
print(1)

1


In [9]:
num_workerssz = 20

# Linear-probe training settings
probe_lr = 1e-3
probe_weight_decay = 1e-4
probe_epochs = 50
patience = 10
PROBE_SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

print(1)


1


In [10]:
train_set_data = ChalearnDataset(
    fullshot=train_set_data[0],
    tagdata=train_set_data[1],
    transform=transform
)

val_set_data = ChalearnDataset(
    fullshot=validation_set_data[0],
    tagdata=validation_set_data[1],
    transform=transform
)

print(1)


1


In [11]:
def extract_ocean(data,batch_size):
    o=[]
    c=[]
    e=[]
    a=[]
    n=[]
    if batch_size >1:
        for i in range(batch_size):
            o.append(data[i][0].item())
            c.append(data[i][1].item())
            e.append(data[i][2].item())
            a.append(data[i][3].item())
            n.append(data[i][4].item())
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n) 
    elif batch_size == 1:
        data=data.squeeze()
        for i in range(batch_size):
            o.append(data[0].item())
            c.append(data[1].item())
            e.append(data[2].item())
            a.append(data[3].item())
            n.append(data[4].item())
            
        o = torch.tensor(o) 
        c = torch.tensor(c) 
        e = torch.tensor(e) 
        a = torch.tensor(a) 
        n = torch.tensor(n)
    return o,c,e,a,n

In [12]:
# ============================================================
# ViViT Video-Only Block-wise Linear Probe
# ============================================================
#
# IMPORTANT:
# 1. This notebook DOES NOT retrain the ViViT backbone.
# 2. It loads the trained video-only ViViT checkpoint.
# 3. The entire backbone is frozen.
# 4. Only one independent nn.Linear probe is trained for each
#    ViViT Transformer block.
# ============================================================

BACKBONE_CHECKPOINT = "./save_swintransformer_folder/model_best_multiseed.pth"

set_seed(PROBE_SEED)

generator = torch.Generator()
generator.manual_seed(PROBE_SEED)

train_dataloader = DataLoader(
    dataset=train_set_data,
    batch_size=batchsz,
    shuffle=True,
    num_workers=num_workerssz,
    worker_init_fn=seed_worker,
    generator=generator
)

val_dataloader = DataLoader(
    dataset=val_set_data,
    batch_size=batchsz,
    shuffle=False,
    num_workers=num_workerssz,
    worker_init_fn=seed_worker
)

# ------------------------------------------------------------
# 1. Build exactly the same ViViT video-only backbone
# ------------------------------------------------------------
backbone = ViViTBackbone(
    2, 14, 14, 2, 2, 2, 5, 96, 6, 10, 8, model=3
).to(device)

checkpoint_data = torch.load(
    BACKBONE_CHECKPOINT,
    map_location=device
)

state_dict = (
    checkpoint_data["model"]
    if isinstance(checkpoint_data, dict) and "model" in checkpoint_data
    else checkpoint_data
)

backbone.load_state_dict(state_dict, strict=True)

# ------------------------------------------------------------
# 2. Frozen backbone + independent linear probes
# ------------------------------------------------------------
model = ViViTLinearProbe(
    backbone=backbone,
    dim=96,
    num_outputs=5
).to(device)

backbone_trainable = sum(
    p.numel()
    for p in model.backbone.parameters()
    if p.requires_grad
)

probe_trainable = sum(
    p.numel()
    for p in model.probe_heads.parameters()
    if p.requires_grad
)

print(f"Backbone trainable parameters: {backbone_trainable}")
print(f"Probe trainable parameters:    {probe_trainable}")

assert backbone_trainable == 0, "Backbone must remain frozen."

criterion = nn.L1Loss().to(device)

optimizer = torch.optim.AdamW(
    model.probe_heads.parameters(),
    lr=probe_lr,
    weight_decay=probe_weight_decay
)

probe_names = list(model.probe_heads.keys())

best_mae = {
    name: float("inf")
    for name in probe_names
}

best_epoch = {
    name: -1
    for name in probe_names
}

# Global early stopping is based on the mean validation MAE
# across all block probes.
best_mean_val_mae = float("inf")
no_improve_count = 0

for epoch in range(probe_epochs):

    # ========================================================
    # Train probes only
    # ========================================================
    model.train()

    train_loss_sum = {
        name: 0.0
        for name in probe_names
    }

    for fullshot, big_five_data in tqdm(
        train_dataloader,
        desc=f"Probe Train {epoch + 1}/{probe_epochs}"
    ):
        fullshot = fullshot.permute(0, 4, 1, 2, 3)
        big_five_data = big_five_data.permute(0, 2, 1).squeeze()

        fullshot = fullshot.to(device)
        big_five_data = big_five_data.to(device)

        optimizer.zero_grad()

        outputs = model(fullshot)

        losses = {
            name: criterion(outputs[name], big_five_data)
            for name in probe_names
        }

        # The heads are independent. Summing their losses is
        # equivalent to optimizing all heads in one backward pass.
        total_loss = sum(losses.values())
        total_loss.backward()
        optimizer.step()

        for name in probe_names:
            train_loss_sum[name] += losses[name].item()

    train_mae = {
        name: train_loss_sum[name] / len(train_dataloader)
        for name in probe_names
    }

    # ========================================================
    # Validation
    # ========================================================
    model.eval()

    val_abs_sum = {
        name: 0.0
        for name in probe_names
    }

    val_count = 0

    with torch.no_grad():

        for fullshot, big_five_data in tqdm(
            val_dataloader,
            desc=f"Probe Valid {epoch + 1}/{probe_epochs}"
        ):
            fullshot = fullshot.permute(0, 4, 1, 2, 3)
            big_five_data = big_five_data.permute(0, 2, 1).squeeze()

            fullshot = fullshot.to(device)
            big_five_data = big_five_data.to(device)

            outputs = model(fullshot)

            batch_count = big_five_data.numel()
            val_count += batch_count

            for name in probe_names:
                val_abs_sum[name] += torch.abs(
                    outputs[name] - big_five_data
                ).sum().item()

    val_mae = {
        name: val_abs_sum[name] / val_count
        for name in probe_names
    }

    # --------------------------------------------------------
    # Each block keeps its own best validation MAE / epoch.
    # --------------------------------------------------------
    for name in probe_names:
        if val_mae[name] < best_mae[name]:
            best_mae[name] = val_mae[name]
            best_epoch[name] = epoch + 1

    mean_val_mae = np.mean(
        [val_mae[name] for name in probe_names]
    )

    if mean_val_mae < best_mean_val_mae:
        best_mean_val_mae = mean_val_mae
        no_improve_count = 0
    else:
        no_improve_count += 1

    print("\n" + "-" * 80)
    print(
        f"Epoch {epoch + 1:02d} | "
        f"Mean Val MAE = {mean_val_mae:.6f} | "
        f"Patience = {no_improve_count}/{patience}"
    )

    for name in probe_names:
        print(
            f"{name:<8} | "
            f"train MAE {train_mae[name]:.6f} | "
            f"val MAE {val_mae[name]:.6f} | "
            f"1-MAE {1.0 - val_mae[name]:.6f} | "
            f"best 1-MAE {1.0 - best_mae[name]:.6f}"
        )

    if no_improve_count >= patience:
        print(
            f"\nEarly stopping at epoch {epoch + 1}. "
            f"Best mean validation MAE = {best_mean_val_mae:.6f}"
        )
        break

    torch.cuda.empty_cache()


# ============================================================
# Final Probe Summary
# ============================================================

print("\n" + "=" * 80)
print("FINAL ViViT VIDEO-ONLY LINEAR PROBE RESULTS")
print("=" * 80)
print(
    f"{'Representation':<18}"
    f"{'Best Epoch':>12}"
    f"{'MAE':>12}"
    f"{'1-MAE':>12}"
)
print("-" * 80)

for name in probe_names:
    print(
        f"{name:<18}"
        f"{best_epoch[name]:>12d}"
        f"{best_mae[name]:>12.6f}"
        f"{1.0 - best_mae[name]:>12.6f}"
    )

print("=" * 80)

# Cleanup
del model
del backbone
del optimizer
del train_dataloader
del val_dataloader

gc.collect()
torch.cuda.empty_cache()


Backbone trainable parameters: 0
Probe trainable parameters:    4062


Probe Valid 1/50: 100%|██████████| 499/499 [00:40<00:00, 12.44it/s]



--------------------------------------------------------------------------------
Epoch 01 | Mean Val MAE = 0.083378 | Patience = 0/10
block1   | train MAE 0.030311 | val MAE 0.082965 | 1-MAE 0.917035 | best 1-MAE 0.917035
block2   | train MAE 0.030705 | val MAE 0.083233 | 1-MAE 0.916767 | best 1-MAE 0.916767
block3   | train MAE 0.032193 | val MAE 0.083052 | 1-MAE 0.916948 | best 1-MAE 0.916948
block4   | train MAE 0.026156 | val MAE 0.083539 | 1-MAE 0.916461 | best 1-MAE 0.916461
block5   | train MAE 0.026750 | val MAE 0.083367 | 1-MAE 0.916633 | best 1-MAE 0.916633
block6   | train MAE 0.028489 | val MAE 0.084113 | 1-MAE 0.915887 | best 1-MAE 0.915887


Probe Valid 2/50: 100%|██████████| 499/499 [00:40<00:00, 12.42it/s]



--------------------------------------------------------------------------------
Epoch 02 | Mean Val MAE = 0.083444 | Patience = 1/10
block1   | train MAE 0.022296 | val MAE 0.083780 | 1-MAE 0.916220 | best 1-MAE 0.917035
block2   | train MAE 0.021907 | val MAE 0.083376 | 1-MAE 0.916624 | best 1-MAE 0.916767
block3   | train MAE 0.021848 | val MAE 0.083295 | 1-MAE 0.916705 | best 1-MAE 0.916948
block4   | train MAE 0.021969 | val MAE 0.083665 | 1-MAE 0.916335 | best 1-MAE 0.916461
block5   | train MAE 0.021993 | val MAE 0.083405 | 1-MAE 0.916595 | best 1-MAE 0.916633
block6   | train MAE 0.021718 | val MAE 0.083144 | 1-MAE 0.916856 | best 1-MAE 0.916856


...


--------------------------------------------------------------------------------
Epoch 34 | Mean Val MAE = 0.082815 | Patience = 8/10
block1   | train MAE 0.018077 | val MAE 0.082798 | 1-MAE 0.917202 | best 1-MAE 0.917585
block2   | train MAE 0.018049 | val MAE 0.082658 | 1-MAE 0.917342 | best 1-MAE 0.917520
block3   | train MAE 0.018038 | val MAE 0.082780 | 1-MAE 0.917220 | best 1-MAE 0.917433
block4   | train MAE 0.018123 | val MAE 0.082915 | 1-MAE 0.917085 | best 1-MAE 0.917467
block5   | train MAE 0.018027 | val MAE 0.082911 | 1-MAE 0.917089 | best 1-MAE 0.917493
block6   | train MAE 0.018114 | val MAE 0.082830 | 1-MAE 0.917170 | best 1-MAE 0.917444


Probe Valid 35/50: 100%|██████████| 499/499 [01:04<00:00,  7.70it/s]



--------------------------------------------------------------------------------
Epoch 35 | Mean Val MAE = 0.082661 | Patience = 9/10
block1   | train MAE 0.018009 | val MAE 0.082729 | 1-MAE 0.917271 | best 1-MAE 0.917585
block2   | train MAE 0.018039 | val MAE 0.082648 | 1-MAE 0.917352 | best 1-MAE 0.917520
block3   | train MAE 0.018034 | val MAE 0.082739 | 1-MAE 0.917261 | best 1-MAE 0.917433
block4   | train MAE 0.018084 | val MAE 0.082679 | 1-MAE 0.917321 | best 1-MAE 0.917467
block5   | train MAE 0.018044 | val MAE 0.082571 | 1-MAE 0.917429 | best 1-MAE 0.917493
block6   | train MAE 0.018027 | val MAE 0.082599 | 1-MAE 0.917401 | best 1-MAE 0.917444


Probe Valid 36/50: 100%|██████████| 499/499 [01:01<00:00,  8.09it/s]


--------------------------------------------------------------------------------
Epoch 36 | Mean Val MAE = 0.082595 | Patience = 10/10
block1   | train MAE 0.018072 | val MAE 0.082519 | 1-MAE 0.917481 | best 1-MAE 0.917585
block2   | train MAE 0.018039 | val MAE 0.082529 | 1-MAE 0.917471 | best 1-MAE 0.917520
block3   | train MAE 0.017999 | val MAE 0.082730 | 1-MAE 0.917270 | best 1-MAE 0.917433
block4   | train MAE 0.018006 | val MAE 0.082563 | 1-MAE 0.917437 | best 1-MAE 0.917467
block5   | train MAE 0.018008 | val MAE 0.082583 | 1-MAE 0.917417 | best 1-MAE 0.917493
block6   | train MAE 0.018075 | val MAE 0.082645 | 1-MAE 0.917355 | best 1-MAE 0.917444

Early stopping at epoch 36. Best mean validation MAE = 0.082551

FINAL ViViT VIDEO-ONLY LINEAR PROBE RESULTS
Representation      Best Epoch         MAE       1-MAE
--------------------------------------------------------------------------------
block1                      26    0.082415    0.917585
block2                      26    0